ұяшық 8.1. Test жиынындағы қателер талдауы

In [2]:
# Ұяшық 8.1 — test жиынындағы қателер талдауы (conf = val-да таңдалған шек)
import json
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

PROJECT_ROOT = Path(r"D:\road_damage_yolo")
BEST = PROJECT_ROOT / "runs" / "main_yolo26n_640_e30_s42" / "weights" / "best.pt"
DATA_YAML = PROJECT_ROOT / "configs" / "data.yaml"
DS = PROJECT_ROOT / "data" / "yolo_dataset"
TAB_DIR = PROJECT_ROOT / "results" / "tables"
THR = json.loads((PROJECT_ROOT / "configs" / "threshold.json").read_text(encoding="utf-8"))["conf"]
NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
print("conf шегі:", THR)

def read_gt(img_path, w, h):
    lbl = DS / "labels" / "test" / (img_path.stem + ".txt")
    boxes = []
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            p = line.split()
            if len(p) != 5:
                continue
            c = int(p[0]); xc, yc, bw, bh = map(float, p[1:])
            boxes.append((c, (xc - bw/2)*w, (yc - bh/2)*h, (xc + bw/2)*w, (yc + bh/2)*h))
    return boxes

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union > 0 else 0.0

model = YOLO(str(BEST))
paths = sorted((DS / "images" / "test").glob("*.jpg"))
pred_rows, gt_rows, bg_images = [], [], []

for p in paths:                                   # бір-бірден, batch = 1
    r = model.predict(str(p), conf=THR, imgsz=640, device=0, verbose=False)[0]
    h, w = r.orig_shape
    gt = read_gt(p, w, h)
    if not gt:
        bg_images.append(p.name)
    pr = sorted(zip(r.boxes.cls.int().tolist(), r.boxes.conf.tolist(), r.boxes.xyxy.tolist()),
                key=lambda t: -t[1])
    used = set()
    for c, cf, bb in pr:
        best, bj = 0.0, -1
        for j, (gc, *gb) in enumerate(gt):
            if j in used or gc != c:
                continue
            v = iou(bb, gb)
            if v > best:
                best, bj = v, j
        if best >= 0.5:
            used.add(bj); kind = "TP"
        else:
            iou_same = max([iou(bb, gb) for gc, *gb in gt if gc == c], default=0.0)
            iou_other = max([iou(bb, gb) for gc, *gb in gt if gc != c], default=0.0)
            if iou_other >= 0.5:
                kind = "FP_класс"
            elif iou_same >= 0.1:
                kind = "FP_локализация/қайталау"
            elif iou_other >= 0.1:
                kind = "FP_класс+локализация"
            else:
                kind = "FP_фон"
        pred_rows.append({"image": p.name, "cls": NAMES[c], "conf": cf,
                          "kind": kind, "bg_image": not gt})
    for j, (gc, x1, y1, x2, y2) in enumerate(gt):
        gt_rows.append({"image": p.name, "cls": NAMES[gc],
                        "area_frac": (x2 - x1) * (y2 - y1) / (w * h), "found": j in used})

P = pd.DataFrame(pred_rows); G = pd.DataFrame(gt_rows)
tp = int((P.kind == "TP").sum()); fp = int((P.kind != "TP").sum()); fn = int((~G.found).sum())
print(f"\nСурет: {len(paths)} | GT: {len(G)} | болжам: {len(P)} | TP {tp}, FP {fp}, FN {fn}")
print(f"Микро-P = {tp/(tp+fp):.3f}, микро-R = {tp/(tp+fn):.3f}  (IoU≥0.5, жеңілдетілген сәйкестендіру)")

print("\n1) Болжам түрлері, класс бойынша:")
print(pd.crosstab(P.cls, P.kind, margins=True, margins_name="барлығы"))
print("\nFP түрлерінің үлесі:")
print(P[P.kind != "TP"].kind.value_counts(normalize=True).round(3).to_string())

print("\n2) Recall (табылу үлесі) класс бойынша:")
print(G.groupby("cls").found.agg(GT_саны="count", recall="mean").round(3).to_string())
G["өлшем"] = pd.qcut(G.area_frac, 3, labels=["кіші", "орта", "үлкен"])
print("\nRecall қорап өлшемі бойынша (суреттің ауданына қатысты үштен бір бөліктер):")
print(G.groupby("өлшем", observed=True).agg(
    GT_саны=("found", "count"), recall=("found", "mean"),
    ауданның_медианасы=("area_frac", "median")).round(4).to_string())

n_bg_fp = P[P.bg_image].image.nunique()
print(f"\n3) Ақаусыз суреттер: {len(bg_images)}, оның ішінде кемінде 1 FP бар: "
      f"{n_bg_fp} ({n_bg_fp/len(bg_images):.1%})")

P.to_csv(TAB_DIR / "test_errors_predictions.csv", index=False)
G.to_csv(TAB_DIR / "test_errors_groundtruth.csv", index=False)

# 4) Шатасу матрицасы conf = THR шегінде (Ultralytics: IoU>0.45, класқа тәуелсіз сәйкестендіру)
m = model.val(data=str(DATA_YAML), split="test", conf=THR, imgsz=640, batch=16, device=0,
              workers=2, plots=True, verbose=False,
              project=str(PROJECT_ROOT / "runs"), name="test_cm_conf0239", exist_ok=True)
labels = list(NAMES.values()) + ["фон"]
cm = pd.DataFrame(m.confusion_matrix.matrix.astype(int),
                  index=[f"болжам_{l}" for l in labels], columns=[f"шын_{l}" for l in labels])
print("\n4) Шатасу матрицасы (саны), conf =", THR)
print(cm.to_string())
cm.to_csv(TAB_DIR / "test_confusion_matrix_conf0239.csv")
print("\nКестелер сақталды:", TAB_DIR)

conf шегі: 0.239

Сурет: 856 | GT: 1622 | болжам: 1309 | TP 836, FP 473, FN 786
Микро-P = 0.639, микро-R = 0.515  (IoU≥0.5, жеңілдетілген сәйкестендіру)

1) Болжам түрлері, класс бойынша:
kind     FP_класс  FP_класс+локализация  FP_локализация/қайталау  FP_фон   TP  \
cls                                                                             
D00             7                     8                       59      29  158   
D10             4                     8                       58      64  172   
D20             6                     9                      109      24  391   
D40             0                     5                       43      40  115   
барлығы        17                    30                      269     157  836   

kind     барлығы  
cls               
D00          261  
D10          306  
D20          539  
D40          203  
барлығы     1309  

FP түрлерінің үлесі:
kind
FP_локализация/қайталау    0.569
FP_фон                     0.332
FP_класс+локализац